In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
cd/content/drive/MyDrive/mc-med_data/physionet.org/files/mc-med/1.0.0

/content/drive/MyDrive/mc-med_data/physionet.org/files/mc-med/1.0.0


In [3]:
cd data

/content/drive/MyDrive/mc-med_data/physionet.org/files/mc-med/1.0.0/data


In [4]:
import pandas as pd
import numpy as np
import re
from datetime import timedelta

In [22]:
df_visits = pd.read_csv('visits.csv')
df_history = pd.read_csv('/content/drive/MyDrive/dx_history_dataset.csv')

In [23]:
df_merged = pd.merge(df_visits, df_history, on='MRN', how='left')

In [24]:
def split_clean_icd(cell):
    """
    Splits a cell of ICD codes into a list, removes leading/trailing spaces,
    and converts to uppercase. Periods within the codes are preserved.
    """
    if pd.isna(cell):
        return []
    parts = re.split(r'[\s;]+', str(cell))
    return [p.strip().upper() for p in parts if p.strip()]

def adjust_time_by_subtracting_year(time_series):
    s = time_series.astype("string")
    year = pd.to_numeric(s.str[:4], errors='coerce')
    month_day_time = s.str[4:]
    new_year = year - 200
    new_time_str = new_year.astype('Int64').astype('string') + month_day_time
    new_time_str = new_time_str.where(s.notna(), pd.NA)
    return pd.to_datetime(new_time_str, errors='coerce').dt.tz_localize(None)

In [33]:
def add_score_CCI(df):
    """Calculates the Charlson Comorbidity Index (CCI)."""
    conditions = [
        (df['Age'] < 50),
        (df['Age'] >= 50) & (df['Age'] <= 59),
        (df['Age'] >= 60) & (df['Age'] <= 69),
        (df['Age'] >= 70) & (df['Age'] <= 79),
        (df['Age'] >= 80)
    ]
    values = [0, 1, 2, 3, 4]
    df['score_CCI_age'] = np.select(conditions, values)

    comorbidity_cols = [
        'cci_MI', 'cci_CHF', 'cci_PVD', 'cci_Stroke', 'cci_Dementia',
        'cci_Pulmonary', 'cci_PUD', 'cci_Rheumatic'
    ]

    df['score_CCI'] = (df['score_CCI_age'] + df[comorbidity_cols].sum(axis=1) +
                       df['cci_Liver1']*1 + df['cci_Liver2']*3 +
                       df['cci_DM1']*1 + df['cci_DM2']*2 +
                       df['cci_Paralysis']*2 + df['cci_Renal']*2 +
                       df['cci_Cancer1']*2 + df['cci_Cancer2']*6 +
                       df['cci_HIV']*6)
    print("Variable 'score_CCI' successfully added")

def add_triage_MAP(df):
    """Calculates and adds the Mean Arterial Pressure (MAP) to the DataFrame."""
    df['Triage_MAP'] = (df['Triage_SBP'] * 1/3) + (df['Triage_DBP'] * 2/3)
    print("Variable 'Triage_MAP' successfully added")

def add_score_REMS(df):
    """Calculates and adds the REMS score."""
    conditions1 = [
        (df['Age'] < 45), (df['Age'] >= 45) & (df['Age'] <= 54),
        (df['Age'] >= 55) & (df['Age'] <= 64), (df['Age'] >= 65) & (df['Age'] <= 74),
        (df['Age'] > 74)
    ]
    values1 = [0, 2, 3, 5, 6]
    conditions2 = [
        (df['Triage_MAP'] > 159), (df['Triage_MAP'] >= 130) & (df['Triage_MAP'] <= 159),
        (df['Triage_MAP'] >= 110) & (df['Triage_MAP'] <= 129), (df['Triage_MAP'] >= 70) & (df['Triage_MAP'] <= 109),
        (df['Triage_MAP'] >= 50) & (df['Triage_MAP'] <= 69), (df['Triage_MAP'] < 49)
    ]
    values2 = [4, 3, 2, 0, 2, 4]
    conditions3 = [
        (df['Triage_HR'] > 179), (df['Triage_HR'] >= 140) & (df['Triage_HR'] <= 179),
        (df['Triage_HR'] >= 110) & (df['Triage_HR'] <= 139), (df['Triage_HR'] >= 70) & (df['Triage_HR'] <= 109),
        (df['Triage_HR'] >= 55) & (df['Triage_HR'] <= 69), (df['Triage_HR'] >= 40) & (df['Triage_HR'] <= 54),
        (df['Triage_HR'] < 40)
    ]
    values3 = [4, 3, 2, 0, 2, 3, 4]
    conditions4 = [
        (df['Triage_RR'] > 49), (df['Triage_RR'] >= 35) & (df['Triage_RR'] <= 49),
        (df['Triage_RR'] >= 25) & (df['Triage_RR'] <= 34), (df['Triage_RR'] >= 12) & (df['Triage_RR'] <= 24),
        (df['Triage_RR'] >= 10) & (df['Triage_RR'] <= 11), (df['Triage_RR'] >= 6) & (df['Triage_RR'] <= 9),
        (df['Triage_RR'] < 6)
    ]
    values4 = [4, 3, 1, 0, 1, 2, 4]
    conditions5 = [
        (df['Triage_SpO2'] < 75), (df['Triage_SpO2'] >= 75) & (df['Triage_SpO2'] <= 85),
        (df['Triage_SpO2'] >= 86) & (df['Triage_SpO2'] <= 89), (df['Triage_SpO2'] > 89)
    ]
    values5 = [4, 3, 1, 0]
    df['score_REMS'] = (np.select(conditions1, values1) + np.select(conditions2, values2) +
                        np.select(conditions3, values3) + np.select(conditions4, values4) +
                        np.select(conditions5, values5))
    print("Variable 'score_REMS' successfully added")

def add_score_CART(df):
    """Calculates and adds the CART score."""
    conditions1 = [
        (df['Age'] < 55), (df['Age'] >= 55) & (df['Age'] <= 69), (df['Age'] >= 70)
    ]
    values1 = [0, 4, 9]
    conditions2 = [
        (df['Triage_RR'] < 21), (df['Triage_RR'] >= 21) & (df['Triage_RR'] <= 23),
        (df['Triage_RR'] >= 24) & (df['Triage_RR'] <= 25), (df['Triage_RR'] >= 26) & (df['Triage_RR'] <= 29),
        (df['Triage_RR'] >= 30)
    ]
    values2 = [0, 8, 12, 15, 22]
    conditions3 = [
        (df['Triage_HR'] < 110), (df['Triage_HR'] >= 110) & (df['Triage_HR'] <= 139),
        (df['Triage_HR'] >= 140)
    ]
    values3 = [0, 4, 13]
    conditions4 = [
        (df['Triage_DBP'] > 49), (df['Triage_DBP'] >= 40) & (df['Triage_DBP'] <= 49),
        (df['Triage_DBP'] >= 35) & (df['Triage_DBP'] <= 39), (df['Triage_DBP'] < 35)
    ]
    values4 = [0, 4, 6, 13]
    df['score_CART'] = (np.select(conditions1, values1) + np.select(conditions2, values2) +
                        np.select(conditions3, values3) + np.select(conditions4, values4))
    print("Variable 'score_CART' successfully added")

def add_score_NEWS(df):
    """Calculates and adds the NEWS score."""
    conditions1 = [
        (df['Triage_RR'] <= 8), (df['Triage_RR'] >= 9) & (df['Triage_RR'] <= 11),
        (df['Triage_RR'] >= 12) & (df['Triage_RR'] <= 20), (df['Triage_RR'] >= 21) & (df['Triage_RR'] <= 24),
        (df['Triage_RR'] >= 25)
    ]
    values1 = [3, 1, 0, 2, 3]
    conditions2 = [
        (df['Triage_SpO2'] <= 91), (df['Triage_SpO2'] >= 92) & (df['Triage_SpO2'] <= 93),
        (df['Triage_SpO2'] >= 94) & (df['Triage_SpO2'] <= 95), (df['Triage_SpO2'] >= 96)
    ]
    values2 = [3, 2, 1, 0]
    conditions3 = [
        (df['Triage_Temp'] <= 35), (df['Triage_Temp'] > 35) & (df['Triage_Temp'] <= 36),
        (df['Triage_Temp'] > 36) & (df['Triage_Temp'] <= 38), (df['Triage_Temp'] > 38) & (df['Triage_Temp'] <= 39),
        (df['Triage_Temp'] > 39)
    ]
    values3 = [3, 1, 0, 1, 2]
    conditions4 = [
        (df['Triage_SBP'] <= 90), (df['Triage_SBP'] >= 91) & (df['Triage_SBP'] <= 100),
        (df['Triage_SBP'] >= 101) & (df['Triage_SBP'] <= 110), (df['Triage_SBP'] >= 111) & (df['Triage_SBP'] <= 219),
        (df['Triage_SBP'] > 219)
    ]
    values4 = [3, 2, 1, 0, 3]
    conditions5 = [
        (df['Triage_HR'] <= 40), (df['Triage_HR'] >= 41) & (df['Triage_HR'] <= 50),
        (df['Triage_HR'] >= 51) & (df['Triage_HR'] <= 90), (df['Triage_HR'] >= 91) & (df['Triage_HR'] <= 110),
        (df['Triage_HR'] >= 111) & (df['Triage_HR'] <= 130), (df['Triage_HR'] > 130)
    ]
    values5 = [3, 1, 0, 1, 2, 3]
    df['score_NEWS'] = (np.select(conditions1, values1) + np.select(conditions2, values2) +
                        np.select(conditions3, values3) + np.select(conditions4, values4) +
                        np.select(conditions5, values5))
    print("Variable 'score_NEWS' successfully added")

def add_score_NEWS2(df):
    """Calculates and adds the NEWS2 score."""
    conditions1 = [
        (df['Triage_RR'] <= 8), (df['Triage_RR'] >= 9) & (df['Triage_RR'] <= 11),
        (df['Triage_RR'] >= 12) & (df['Triage_RR'] <= 20), (df['Triage_RR'] >= 21) & (df['Triage_RR'] <= 24),
        (df['Triage_RR'] >= 25)
    ]
    values1 = [3, 1, 0, 2, 3]
    conditions2 = [
        (df['Triage_Temp'] <= 35), (df['Triage_Temp'] > 35) & (df['Triage_Temp'] <= 36),
        (df['Triage_Temp'] > 36) & (df['Triage_Temp'] <= 38), (df['Triage_Temp'] > 38) & (df['Triage_Temp'] <= 39),
        (df['Triage_Temp'] > 39)
    ]
    values2 = [3, 1, 0, 1, 2]
    conditions3 = [
        (df['Triage_SBP'] <= 90), (df['Triage_SBP'] >= 91) & (df['Triage_SBP'] <= 100),
        (df['Triage_SBP'] >= 101) & (df['Triage_SBP'] <= 110), (df['Triage_SBP'] >= 111) & (df['Triage_SBP'] <= 219),
        (df['Triage_SBP'] > 219)
    ]
    values3 = [3, 2, 1, 0, 3]
    conditions4 = [
        (df['Triage_HR'] <= 40), (df['Triage_HR'] >= 41) & (df['Triage_HR'] <= 50),
        (df['Triage_HR'] >= 51) & (df['Triage_HR'] <= 90), (df['Triage_HR'] >= 91) & (df['Triage_HR'] <= 110),
        (df['Triage_HR'] >= 111) & (df['Triage_HR'] <= 130), (df['Triage_HR'] > 130)
    ]
    values4 = [3, 1, 0, 1, 2, 3]
    df['score_NEWS2'] = (np.select(conditions1, values1) + np.select(conditions2, values2) +
                         np.select(conditions3, values3) + np.select(conditions4, values4))
    print("Variable 'score_NEWS2' successfully added")

def add_score_MEWS(df):
    """Calculates and adds the MEWS score."""
    conditions1 = [
        (df['Triage_SBP'] <= 70), (df['Triage_SBP'] >= 71) & (df['Triage_SBP'] <= 80),
        (df['Triage_SBP'] >= 81) & (df['Triage_SBP'] <= 100), (df['Triage_SBP'] >= 101) & (df['Triage_SBP'] <= 199),
        (df['Triage_SBP'] > 199)
    ]
    values1 = [3, 2, 1, 0, 2]
    conditions2 = [
        (df['Triage_HR'] <= 40), (df['Triage_HR'] >= 41) & (df['Triage_HR'] <= 50),
        (df['Triage_HR'] >= 51) & (df['Triage_HR'] <= 100), (df['Triage_HR'] >= 101) & (df['Triage_HR'] <= 110),
        (df['Triage_HR'] >= 111) & (df['Triage_HR'] <= 129), (df['Triage_HR'] >= 130)
    ]
    values2 = [2, 1, 0, 1, 2, 3]
    conditions3 = [
        (df['Triage_RR'] < 9), (df['Triage_RR'] >= 9) & (df['Triage_RR'] <= 14),
        (df['Triage_RR'] >= 15) & (df['Triage_RR'] <= 20), (df['Triage_RR'] >= 21) & (df['Triage_RR'] <= 29),
        (df['Triage_RR'] >= 30)
    ]
    values3 = [2, 0, 1, 2, 3]
    conditions4 = [
        (df['Triage_Temp'] < 35), (df['Triage_Temp'] >= 35) & (df['Triage_Temp'] < 38.5),
        (df['Triage_Temp'] >= 38.5)
    ]
    values4 = [2, 0, 2]
    df['score_MEWS'] = (np.select(conditions1, values1) + np.select(conditions2, values2) +
                        np.select(conditions3, values3) + np.select(conditions4, values4))
    print("Variable 'score_MEWS' successfully added")

add_triage_MAP(df_merged)
add_score_REMS(df_merged)
add_score_CART(df_merged)
add_score_NEWS(df_merged)
add_score_NEWS2(df_merged)
add_score_MEWS(df_merged)
add_score_CCI(df_merged)


print("\n--- Calculated Scores for Each Visit ---")
score_columns = ['MRN', 'score_CCI', 'score_CART','score_REMS',  'score_NEWS', 'score_NEWS2', 'score_MEWS']
print(df_merged[score_columns])


Variable 'Triage_MAP' successfully added
Variable 'score_REMS' successfully added
Variable 'score_CART' successfully added
Variable 'score_NEWS' successfully added
Variable 'score_NEWS2' successfully added
Variable 'score_MEWS' successfully added
Variable 'score_CCI' successfully added

--- Calculated Scores for Each Visit ---
             MRN  score_CCI  score_CART  score_REMS  score_NEWS  score_NEWS2  \
0       98880961          5           9           6           0            0   
1       99121566          8           4           3           2            2   
2       99121566          8           4           3           2            2   
3       99121566          9           4           3           3            3   
4       99121566          9           4           3           3            3   
...          ...        ...         ...         ...         ...          ...   
471188  99648524          0           0           0           0            0   
471189  99312340          0    